In [ ]:
import json
import csv
import re
import os

def parse_arabic_text(text):
    """
    Parses the structured text (Story, Question, Options) from the
    refined_arabic or initial_arabic fields.

    Args:
        text (str): The string containing the story, question, and options,
                    potentially wrapped in markdown code blocks.

    Returns:
        tuple: A tuple containing (story, question, options_dict),
               or (None, None, {}) if parsing fails.
    """
    if not text or not isinstance(text, str):
        return None, None, {}

    # Remove potential markdown code block fences
    text = re.sub(r'^```[a-zA-Z]*\n', '', text, flags=re.MULTILINE)
    text = re.sub(r'\n```$', '', text, flags=re.MULTILINE)
    # Remove potential start/end markers like '---'
    text = re.sub(r'^---\n', '', text, flags=re.MULTILINE)
    text = re.sub(r'\n---\n?$', '', text, flags=re.MULTILINE)
    text = text.strip()

    story = None
    question = None
    options_dict = {}

    try:
        # Use regex to find sections robustly, allowing for variations in spacing/newlines
        story_match = re.search(r'###STORY\s*(.*?)\s*###QUESTION', text, re.DOTALL | re.IGNORECASE)
        question_match = re.search(r'###QUESTION\s*(.*?)\s*###OPTIONS', text, re.DOTALL | re.IGNORECASE)
        options_match = re.search(r'###OPTIONS\s*(.*)', text, re.DOTALL | re.IGNORECASE)

        if story_match:
            story = story_match.group(1).strip()

        if question_match:
            question = question_match.group(1).strip()

        if options_match:
            options_text = options_match.group(1).strip()
            # Regex to capture options like (A) Text, (B) Text, etc.
            # Handles potential variations in spacing after the letter.
            option_lines = re.findall(r'^\s*\(([A-Z])\)\s*(.*)', options_text, re.MULTILINE | re.IGNORECASE)
            for letter, opt_text in option_lines:
                options_dict[letter.upper()] = opt_text.strip()

    except Exception as e:
        print(f"Error parsing text block: {e}\nContent: {text[:200]}...") # Print first 200 chars on error
        return None, None, {}

    return story, question, options_dict

def convert_jsonl_folder_to_csv(input_folder_path, csv_output_path):
    """
    Converts all JSONL files in a specified folder to a single CSV file.

    Args:
        input_folder_path (str): Path to the folder containing input JSONL files.
        csv_output_path (str): Path to the output CSV file.
    """
    # Define CSV headers
    headers = [
        '#', 'ID', 'Source', 'Country', 'Group', 'Subject', 'Level',
        'Question', 'BackStory', 'Context', 'Answer Key', 'Option 1',
        'Option 2', 'Option 3', 'Option 4', 'Option 5', 'is_few_shot'
    ]

    global_row_counter = 0 # Counter across all files

    # Check if input folder exists
    if not os.path.isdir(input_folder_path):
        print(f"Error: Input folder not found at '{input_folder_path}'")
        print("In Google Colab, make sure the 'data' folder exists in your current directory,")
        print("or mount your Google Drive and provide the correct path.")
        return

    try:
        with open(csv_output_path, 'w', newline='', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(headers) # Write header row once

            # Iterate through all files in the input folder
            for filename in os.listdir(input_folder_path):
                # Process only files ending with .jsonl
                if filename.lower().endswith('.jsonl'):
                    jsonl_input_path = os.path.join(input_folder_path, filename)
                    print(f"Processing file: {jsonl_input_path}...")

                    try:
                        with open(jsonl_input_path, 'r', encoding='utf-8') as infile:
                            file_line_counter = 0
                            for line in infile:
                                file_line_counter += 1
                                global_row_counter += 1
                                try:
                                    # Load JSON data from each line
                                    data = json.loads(line.strip())

                                    # Extract data - use .get() for safety against missing keys
                                    scenario_id = data.get('scenario_id')

                                    # Get the source file value from the JSON data itself
                                    source_file_from_json = data.get('source_file', '') # Use value from JSON field

                                    # Prioritize refined_arabic, fall back to initial_arabic
                                    arabic_text = data.get('refined_arabic', data.get('initial_arabic'))

                                    # Parse the Arabic text block
                                    story, question, options = parse_arabic_text(arabic_text)

                                    # Attempt to get the answer key, prioritizing the new field name
                                    answer_key_field_name = "答案\nANSWER"
                                    answer_key = data.get(answer_key_field_name,
                                                          data.get('answer_key',
                                                                   data.get('correct_answer', '')))

                                    # --- MODIFICATION START ---
                                    # Extract subject from the source_file value within the JSON
                                    subject = source_file_from_json # Start with the value from JSON
                                    if subject and subject.lower().endswith('.jsonl'): # Check if not empty before checking suffix
                                         # Use removesuffix if available (Python 3.9+) for cleaner removal
                                        try:
                                            subject = subject.removesuffix('.jsonl')
                                        except AttributeError:
                                            # Fallback for older Python versions
                                            subject = subject[:-len('.jsonl')]
                                    # --- MODIFICATION END ---


                                    # Prepare row data, using defaults for missing fields
                                    csv_row = [
                                        global_row_counter,                     # '#' (Global counter)
                                        scenario_id,                            # 'ID'
                                        source_file_from_json,                  # 'Source' (Value from JSON 'source_file' key)
                                        '',                                     # 'Country'
                                        '',                                     # 'Group'
                                        subject,                                # 'Subject' (Value from JSON 'source_file' key without extension)
                                        '',                                     # 'Level'
                                        question,                               # 'Question'
                                        story,                                  # 'BackStory'
                                        '',                                     # 'Context'
                                        answer_key,                             # 'Answer Key' (Extracted value)
                                        options.get('A'),                       # 'Option 1'
                                        options.get('B'),                       # 'Option 2'
                                        options.get('C'),                       # 'Option 3'
                                        options.get('D'),                       # 'Option 4'
                                        options.get('E'),                       # 'Option 5'
                                        False                                   # 'is_few_shot'
                                    ]
                                    writer.writerow(csv_row)

                                except json.JSONDecodeError:
                                    print(f"  Warning: Skipping invalid JSON on line {file_line_counter} in {filename}: {line.strip()}")
                                except Exception as e:
                                    print(f"  Warning: Error processing line {file_line_counter} in {filename}: {e}")
                                    print(f"  Problematic line content: {line.strip()}")
                    except FileNotFoundError:
                         print(f"  Error: Could not open input file '{jsonl_input_path}' (unexpected)")
                    except Exception as e:
                         print(f"  Error reading file {jsonl_input_path}: {e}")

        print(f"\nSuccessfully processed files from '{input_folder_path}' into '{csv_output_path}'.")
        print(f"Total lines written to CSV: {global_row_counter}")

    except IOError as e:
        print(f"Error writing to output file '{csv_output_path}': {e}")
    except Exception as e:
         print(f"An unexpected error occurred during processing: {e}")

# --- Script Execution ---
if __name__ == "__main__":
    # --- Configuration for Google Colab ---
    # 1. Ensure your 'data' folder containing the .jsonl files is accessible.
    #    - You can upload it directly to the Colab environment's file system (temporary).
    #    - Or, mount your Google Drive:
    #      from google.colab import drive
    #      drive.mount('/content/drive')
    #      # Then adjust input_folder to point to your Drive path, e.g.,
    #      # input_folder = '/content/drive/MyDrive/your_project_folder/data'

    # Define input folder and output file paths
    input_folder = 'data'  # Assumes 'data' folder is in the same directory as the script
                           # or in the root of your Colab environment/mounted Drive.
    output_file = 'output.csv' # The combined output CSV file name.

    # Run the conversion function
    convert_jsonl_folder_to_csv(input_folder, output_file)


Processing file: data/output_data_matched_by_story_final (1).jsonl...

Successfully processed files from 'data' into 'output.csv'.
Total lines written to CSV: 2860
